In [ ]:
import re
import os
import matplotlib.pyplot as plt
import numpy as np
import contextily as ctx
from shapely import wkt
from shapely.wkt import loads
from shapely.geometry import mapping
from shapely.ops import unary_union
import geopandas as gpd
import pandas as pd
import h5py
import s3fs
import asf_search as asf
import earthaccess
from rasterio.crs import CRS
from rasterio.transform import Affine
import rasterio
import json
import warnings
import socket, fsspec

# from openseppo.cli import nisar_search

In [ ]:
# parse the filename and the query, to get some information about the data found
def parse_nisar_filename(fileID, s3Urls, https_url=None):
    parts = fileID.split('_')
    date = next(p[:8] for p in parts if re.match(r'\d{8}T\d{6}', p))
    h5_url = next((u for u in s3Urls if re.search(r'\d+\.h5$', u)), None)
    return {
        'product': parts[3],
        'track':   parts[5],
        'orb':     parts[6],
        'frame':   parts[7],
        'date':    date,
        'proc':    parts[13],
        's3_url':  h5_url,
        'https_url': https_url,
    }

In [ ]:
# area of interest for querying data
lat, lon = 29.6, -82.0
west, east, south, north = lon-0.05, lon+0.05, lat-0.05, lat+0.05

# indicate a buffer area around the search, this will make nicer images
delta = 0.3
aoi_search = f'POLYGON(({west} {south}, {east} {south}, {east} {north}, {west} {north}, {west} {south}))'
# delineate the imaging area
west_img = west - delta
east_img = east + delta
south_img = south - delta
north_img = north + delta/2

aois = {
    'Gainesville': f'POLYGON(({west_img:.2f} {south_img:.2f}, {east_img:.2f} {south_img:.2f}, {east_img:.2f} {north_img:.2f}, {west_img:.2f} {north_img:.2f}, {west_img:.2f} {south_img:.2f}))'
}

results = asf.geo_search(
    platform=[asf.PLATFORM.NISAR],
    intersectsWith=aoi_search,
    processingLevel=['GCOV'],
    maxResults=100
)

if len(results) == 0:
    print('No NISAR GCOV data found!')
else:
    print(f"Found {len(results)} GCOV results")
    print(aois)

In [ ]:
# make an optical image of the region along with the area of interest
#
# EPSG 4326 is lat/lon (deg)
# EPSG 3857 is Web Mercator (meters)
geoms = [loads(wkt_str) for wkt_str in aois.values()]
all_geom = unary_union(geoms)
gdf = gpd.GeoDataFrame(geometry=geoms, crs='EPSG:4326')
gdf['name'] = list(aois.keys())
gdf_web = gdf.to_crs(epsg=3857)

buf = 1.0   # buffer around the area of interest (units of degrees)
buffered = gpd.GeoDataFrame(geometry=[all_geom.buffer(buf)], crs='EPSG:4326').to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 8))
gdf_web.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)
for _, r in gdf_web.iterrows():
    ax.annotate(r['name'], xy=r.geometry.centroid.coords[0], ha='center', color='yellow', fontsize=10)

minx, miny, maxx, maxy = buffered.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
plt.title('AOIs')
plt.tight_layout()
plt.show()

In [ ]:
# summary of the data found
# some filtering is used to make it so that there are only ascending or descending orbits included.
parsed = []
for r in results:
    try:
        parsed.append(parse_nisar_filename(
            r.properties['fileID'],
            r.properties['s3Urls'],
            https_url=r.properties['url'],
        ))
    except Exception as e:
        print(f"Could not parse {r.properties['fileID']!r}: {e}")

all_products = sorted({p['product'] for p in parsed})
print(f"Products found: {all_products}")

gcov = [p for p in parsed if p['product'] == 'GCOV']
df = pd.DataFrame(gcov)
if df.empty:
    print("No GCOV results — check filename format or search area.")
else:
    df = df[df['orb'] == 'D']
    df = df.sort_values('date').reset_index(drop=True)
    print(df.drop(columns=['s3_url', 'https_url']).to_string())

In [ ]:
auth = earthaccess.login(strategy='interactive')
print(f"Authenticated: {auth.authenticated}")
print(f"Logged in as:  {auth.username}")

session = auth.get_session()

def _in_aws():
    """Return True if the EC2 instance metadata endpoint is reachable (i.e. running inside AWS)."""
    try:
        socket.setdefaulttimeout(1)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('169.254.169.254', 80))
        return True
    except Exception:
        return False

if _in_aws():
    print("Inside AWS — using direct S3 access")
    endpoint = 'https://nisar.asf.earthdatacloud.nasa.gov/s3credentials'
    s3_credentials = auth.get_s3_credentials(endpoint=endpoint)
    fs_nisar = s3fs.S3FileSystem(
        anon=False,
        key=s3_credentials['accessKeyId'],
        secret=s3_credentials['secretAccessKey'],
        token=s3_credentials['sessionToken'],
        skip_instance_cache=True
    )
    use_url = 's3_url'
else:
    print("I see that you are outside of the AWS, perhaps running on a local computer")
    print("   --> that is okay, but running the notebook will be slower and less efficient")
    print("       consider using an AWS cloud instance in the future")
    print("  — using HTTPS access instead of direct s3 access now")
    fs_nisar = fsspec.filesystem('https', headers=dict(session.headers), skip_instance_cache=True)
    use_url = 'https_url'

# connectivity test
test_url = df.iloc[0][use_url]
try:
    info = fs_nisar.info(test_url.replace('s3://', '') if use_url == 's3_url' else test_url)
    print(f"Access OK — file size: {info['size'] / 1e9:.2f} GB")
except Exception as e:
    print(f"Access FAILED: {e}")

In [ ]:
for idx, row in df.iterrows():
    h5file = row[use_url]
    with fs_nisar.open(h5file, 'rb') as f:
        with h5py.File(f, 'r') as h5:
            proj = h5['science/LSAR/GCOV/grids/frequencyA/projection'][()]
            print(f"{idx:02d}  \tEPSG: {proj} \t{row['date']} \t{row['track']} {row['orb']} {row['frame']}")

In [ ]:
EPSG_keep = 32617
epsg_string = f"EPSG:{EPSG_keep}"

bb_list = []
proj_list = []
keep_mask = []

for idx, row in df.iterrows():
    h5file = row[use_url]
    with fs_nisar.open(h5file, 'rb') as f:
        with h5py.File(f, 'r') as h5:
            proj = h5['science/LSAR/GCOV/grids/frequencyA/projection'][()]
            passes = (proj == EPSG_keep)
            keep_mask.append(passes)
            if passes:
                bb     = wkt.loads(h5['science/LSAR/GCOV/metadata/ceosAnalysisReadyData/boundingBox'])
                xdelta = h5['science/LSAR/GCOV/grids/frequencyA/xCoordinateSpacing'][()]
                ydelta = h5['science/LSAR/GCOV/grids/frequencyA/yCoordinateSpacing'][()]
                bb_list.append(bb)
                proj_list.append(proj)
                print(f"{idx:02d}  {bb}\tEPSG: {proj}\tdx={xdelta:.1f}, dy={ydelta:.1f}")

df = df[keep_mask].reset_index(drop=True)
bb_total = unary_union(bb_list)
minx, miny, maxx, maxy = np.array(bb_list[0].bounds) / 1000.0
print(f"(minx, miny, maxx, maxy): ({minx:.2f}, {miny:.2f}, {maxx:.2f}, {maxy:.2f})")

In [ ]:
stride = 1
all_subsets = {}

for aoi_name, aoi_wkt in aois.items():
    aoi_gdf = gpd.GeoDataFrame(geometry=[loads(aoi_wkt)], crs='EPSG:4326').to_crs(epsg_string)
    aoi_xmin, aoi_ymin, aoi_xmax, aoi_ymax = aoi_gdf.geometry[0].bounds
    print(f"\nAOI '{aoi_name}' in {epsg_string}: x=[{aoi_xmin:.0f}, {aoi_xmax:.0f}], y=[{aoi_ymin:.0f}, {aoi_ymax:.0f}]")

    subsets = {}
    for idx, row in df.iterrows():
        h5file = row[use_url]
        key = f"{row['date']}_t{row['track']}_f{row['frame']}"
        with fs_nisar.open(h5file, 'rb') as f:
            with h5py.File(f, 'r') as h5:
                xCoords = h5['/science/LSAR/GCOV/grids/frequencyA/xCoordinates'][()]
                yCoords = h5['/science/LSAR/GCOV/grids/frequencyA/yCoordinates'][()]

                xi0 = np.searchsorted(xCoords, aoi_xmin)
                xi1 = np.searchsorted(xCoords, aoi_xmax)
                yi0 = np.searchsorted(-yCoords, -aoi_ymax)
                yi1 = np.searchsorted(-yCoords, -aoi_ymin)

                if xi0 >= xi1 or yi0 >= yi1:
                    print(f"  {key}: AOI does not overlap, skipping")
                    continue

                hhhh = h5['/science/LSAR/GCOV/grids/frequencyA/HHHH'][yi0:yi1, xi0:xi1]
                hvhv = h5['/science/LSAR/GCOV/grids/frequencyA/HVHV'][yi0:yi1, xi0:xi1]

                if stride != 1:
                    hhhh = hhhh[::stride, ::stride]
                    hvhv = hvhv[::stride, ::stride]

        subsets[key] = {
            'date':    row['date'],
            'track':   row['track'],
            'frame':   row['frame'],
            'HHHH':    hhhh,
            'HVHV':    hvhv,
            'xCoords': xCoords[xi0:xi1:stride],
            'yCoords': yCoords[yi0:yi1:stride],
        }
        print(f"  {key}: shape={hhhh.shape}")

        with np.errstate(divide='ignore', invalid='ignore'):
            data_db = 10 * np.log10(np.abs(hhhh))
            data_db[~np.isfinite(data_db)] = np.nan

        if False:
            fig, ax = plt.subplots(figsize=(10, 8))
            im = ax.imshow(data_db, cmap='gray',
                           vmin=np.nanpercentile(data_db, 2),
                           vmax=np.nanpercentile(data_db, 98))
            plt.colorbar(im, ax=ax, label='dB')
            ax.set_title(f"{aoi_name} — GCOV HHHH  {row['date']}  track={row['track']}  frame={row['frame']}")
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    all_subsets[aoi_name] = subsets
    print(f"  -> {len(subsets)} subsets for '{aoi_name}'")

print(f"\nDone. Total AOIs: {len(all_subsets)}")

In [ ]:
dynamic_range = 16
hhmax, hhmin = -4,  -4 - dynamic_range
hvmax, hvmin = -8, -8 - dynamic_range

for aoi_name, subsets in all_subsets.items():
    print(f"\n=== {aoi_name} ===")
    for key, s in subsets.items():
        with np.errstate(divide='ignore', invalid='ignore'):
            hhhh_db = 10 * np.log10(np.abs(s['HHHH'])).astype(np.float32)
            hvhv_db = 10 * np.log10(np.abs(s['HVHV'])).astype(np.float32)

        x_km = (s['xCoords'] - s['xCoords'][0]) / 1000.0
        y_km = (s['yCoords'] - s['yCoords'][0]) / 1000.0
        extent = [x_km[0], x_km[-1], y_km[-1], y_km[0]]

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        print(f"[{s['date']} track={s['track']} frame={s['frame']}]")

        for ax, data, label, vmin, vmax in [
            (axes[0], hhhh_db, 'HHHH', hhmin, hhmax),
            (axes[1], hvhv_db, 'HVHV', hvmin, hvmax),
        ]:
            im = ax.imshow(data, cmap='gray', vmin=vmin, vmax=vmax, extent=extent)
            plt.colorbar(im, ax=ax, label='dB', fraction=0.046, pad=0.04)
            ax.set_title(f"{aoi_name} — {label} — {s['date']} t{s['track']} f{s['frame']}")
            ax.set_xlabel('Easting (rel.) (km)')
            ax.set_ylabel('Northing (rel.) (km)')

        plt.tight_layout()
        plt.show()
        plt.close(fig)

In [ ]:
stats_by_aoi = {}
for aoi_name, subsets in all_subsets.items():
    print(f"\n=== Stacking: {aoi_name} ===")
    hhhh_stack = np.stack([s['HHHH'] for s in subsets.values()], axis=0).astype(np.float32)
    hvhv_stack = np.stack([s['HVHV'] for s in subsets.values()], axis=0).astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        hhhh_dBstack = 10 * np.log10(hhhh_stack)
        hvhv_dBstack = 10 * np.log10(hvhv_stack)
    hhhh_dBstack[~np.isfinite(hhhh_dBstack)] = np.nan
    hvhv_dBstack[~np.isfinite(hvhv_dBstack)] = np.nan
    s0 = next(iter(subsets.values()))
    dx = float(s0['xCoords'][1] - s0['xCoords'][0]) if len(s0['xCoords']) > 1 else float(xdelta) * stride
    dy = float(s0['yCoords'][1] - s0['yCoords'][0]) if len(s0['yCoords']) > 1 else float(ydelta) * stride

    with warnings.catch_warnings():   # supresses what happens when there are some no-data values 
        warnings.filterwarnings('ignore', r'All-NaN slice encountered', RuntimeWarning)
        warnings.filterwarnings('ignore', r'Degrees of freedom <= 0', RuntimeWarning)
        warnings.filterwarnings('ignore', r'Mean of empty slice', RuntimeWarning)
        stats_by_aoi[aoi_name] = {
            'hhhh_dBmedian': np.nanmedian(hhhh_dBstack, axis=0),
            'hvhv_dBmedian': np.nanmedian(hvhv_dBstack, axis=0),
            'hhhh_dBstd':    np.nanstd(hhhh_dBstack,    axis=0),
            'hvhv_dBstd':    np.nanstd(hvhv_dBstack,    axis=0),
            'hhhh_median':   np.nanmedian(hhhh_stack, axis=0),
            'hvhv_median':   np.nanmedian(hvhv_stack, axis=0),
            'xCoords': s0['xCoords'],
            'yCoords': s0['yCoords'],
            'dx': dx,
            'dy': dy,
        }
    print(f"  stack shape: {hhhh_stack.shape}")
print(f"\nFinished!  Stats computed for: {list(stats_by_aoi.keys())}")

In [ ]:
def normalize(arr, pmin=2, pmax=98):
    lo, hi = np.nanpercentile(arr, pmin), np.nanpercentile(arr, pmax)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

for aoi_name, st in stats_by_aoi.items():
    rgb = np.dstack([
        normalize(st['hhhh_dBmedian']),
        normalize(st['hvhv_dBmedian']),
        normalize(st['hhhh_dBstd']),
    ])

    x_km = (st['xCoords'] - st['xCoords'][0]) / 1000.0
    y_km = (st['yCoords'] - st['yCoords'][0]) / 1000.0
    extent = [x_km[0], x_km[-1], y_km[-1], y_km[0]]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb, extent=extent)
    ax.set_title(f"{aoi_name} — RGB: R=HHHH median, G=HVHV median, B=HHHH std")
    ax.set_xlabel('Easting (rel.) (km)')
    ax.set_ylabel('Northing (rel.) (km)')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
if False:
    targ_dir = '.'
    crs = CRS.from_epsg(EPSG_keep)

    for aoi_name, st in stats_by_aoi.items():
        dx = st['dx']
        dy = st['dy']
        ulx = float(st['xCoords'][0]) - dx / 2.0
        uly = float(st['yCoords'][0]) - dy / 2.0
        transform = Affine(dx, 0.0, ulx, 0.0, dy, uly)

        h, w = st['hhhh_dBmedian'].shape
        profile = {
            "driver":     "GTiff",
            "height":     h,
            "width":      w,
            "count":      1,
            "dtype":      "float32",
            "crs":        crs,
            "transform":  transform,
            "compress":   "deflate",
            "tiled":      True,
            "blockxsize": 256,
            "blockysize": 256,
        }

        tag = aoi_name.replace(' ', '_')
        with rasterio.open(os.path.join(targ_dir, f"{tag}_HHHH.tif"), "w", **profile) as dst:
            dst.write(st['hhhh_dBmedian'].astype("float32"), 1)
        with rasterio.open(os.path.join(targ_dir, f"{tag}_HVHV.tif"), "w", **profile) as dst:
            dst.write(st['hvhv_dBmedian'].astype("float32"), 1)
        with rasterio.open(os.path.join(targ_dir, f"{tag}_HHstd.tif"), "w", **profile) as dst:
            dst.write(st['hhhh_dBstd'].astype("float32"), 1)
        print(f"Saved GeoTIFFs for '{aoi_name}'")